# 04 数值实验

本节把插值算法放到可测量的实验中。重点不是单独画几张图，而是把算法选择与可观察的误差行为联系起来。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    NaturalCubicSpline,
    chebyshev_nodes,
    lagrange_interpolate,
    piecewise_linear_interpolate,
)


## 实验 1：等距节点与切比雪夫节点

Runge 函数说明，增加等距节点并不一定降低最大误差。切比雪夫节点通常能更好地控制端点附近的振荡。


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_plot = np.linspace(-1.0, 1.0, 1200)
y_true = runge(x_plot)
node_counts = np.arange(5, 32, 2)
errors_equal = []
errors_cheb = []

for n in node_counts:
    x_equal = np.linspace(-1.0, 1.0, n)
    x_cheb = chebyshev_nodes(-1.0, 1.0, n)
    y_equal = lagrange_interpolate(x_equal, runge(x_equal), x_plot)
    y_cheb = lagrange_interpolate(x_cheb, runge(x_cheb), x_plot)
    errors_equal.append(np.max(np.abs(y_equal - y_true)))
    errors_cheb.append(np.max(np.abs(y_cheb - y_true)))

plt.figure(figsize=(8, 4.5))
plt.semilogy(node_counts, errors_equal, "o-", label="等距节点")
plt.semilogy(node_counts, errors_cheb, "o-", label="切比雪夫节点")
plt.xlabel("节点数")
plt.ylabel("最大误差")
plt.title("Runge 实验：节点分布控制误差")
plt.legend()
plt.tight_layout()
plt.show()


## 实验 2：全局多项式、分段线性与样条

对于中等光滑的数据，样条经常比粗糙的分段线性插值和高次全局多项式插值更折中。


In [ ]:
def target(x):
    return np.sin(2.5 * x) + 0.2 * x

x_data = np.linspace(0.0, 3.0, 9)
y_data = target(x_data)
x_eval = np.linspace(0.0, 3.0, 600)
y_exact = target(x_eval)

methods = {
    "全局多项式": lagrange_interpolate(x_data, y_data, x_eval),
    "分段线性": piecewise_linear_interpolate(x_data, y_data, x_eval),
    "自然三次样条": NaturalCubicSpline.fit(x_data, y_data)(x_eval),
}

for name, values in methods.items():
    print(f"{name:10s} 最大误差 = {np.max(np.abs(values - y_exact)):10.3e}")

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, y_exact, color="black", label="目标函数")
for name, values in methods.items():
    plt.plot(x_eval, values, label=name)
plt.scatter(x_data, y_data, color="black", s=18, zorder=3)
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("光滑数据上的方法对比")
plt.legend()
plt.tight_layout()
plt.show()


## 实验 3：含噪声数据下的插值

插值会精确通过数据点。若观测数据含有噪声，这个精确约束可能成为缺点。


In [ ]:
rng = np.random.default_rng(7)
x_noisy = np.linspace(-1.0, 1.0, 13)
y_noisy = runge(x_noisy) + 0.04 * rng.normal(size=x_noisy.size)

poly_noisy = lagrange_interpolate(x_noisy, y_noisy, x_plot)
linear_noisy = piecewise_linear_interpolate(x_noisy, y_noisy, x_plot)
spline_noisy = NaturalCubicSpline.fit(x_noisy, y_noisy)(x_plot)

plt.figure(figsize=(8, 4.5))
plt.plot(x_plot, y_true, color="black", label="真实 Runge 函数")
plt.plot(x_plot, poly_noisy, label="穿过噪声点的全局多项式")
plt.plot(x_plot, linear_noisy, label="分段线性插值")
plt.plot(x_plot, spline_noisy, label="自然三次样条")
plt.scatter(x_noisy, y_noisy, color="black", s=20, zorder=3, label="含噪声数据")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("精确插值会保留并放大噪声")
plt.legend()
plt.tight_layout()
plt.show()


## 实验小结

* 对 Runge 函数，等距节点插值可能随着次数升高而失效。
* 切比雪夫节点能显著改善这个测试中的全局多项式插值。
* 分段方法和样条方法通常更稳健，因为它们具有局部性。
* 遇到噪声数据时，应先判断问题需要插值还是拟合。
